In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Mock display for script
def display(html_obj):
    pass

# Mock HTML for script
class HTML:
    def __init__(self, data):
        self.data = data

MIXER = {
    "RELATIVE_STRENGTH": 0.75,      # Allow more candidates
    "SIMALIGN_THRESHOLD": 0.15,     # Very low for max recall
    "AWESOME_THRESHOLD": 0.05,      # Lowered because of Symmetric Softmax normalization
    "RESCUE_ORPHANS": True,
    "RESCUE_THRESHOLD": 0.15,       # Aggressive rescue
    "WEIGHT_SIMALIGN": 0.50,        # Balanced
    "WEIGHT_AWESOME": 0.50,
    "STRICT_METRIC": False
}

print("⏳ Loading High-Performance Models...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LaBSE is superior for cross-lingual word similarity
sim_model = SentenceTransformer('sentence-transformers/LaBSE').to(device)

awesome_id = "ai4bharat/IndicBERTv2-MLM-Sam-TLM"
awesome_tokenizer = AutoTokenizer.from_pretrained(awesome_id)
awesome_model = AutoModel.from_pretrained(awesome_id, output_hidden_states=True).to(device)
awesome_model.eval()
print("✅ Models Ready.")




⏳ Loading High-Performance Models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

✅ Models Ready.


In [ ]:
# 2. DATA INPUTS
hindi_pairs = [
    ("Also, in the world, when social media started being used for crises, the first time it was used to propagate an incident was different.",
     "इसके अलावा, दुनिया में जब संकटों के दौरान सोशल मीडिया का इस्तेमाल शुरू हुआ, तो पहली बार इसका इस्तेमाल किसी घटना को प्रचारित करने के लिए किया गया था, वह अलग बात थी।"),

    ("For example, the 2009 Hudson River incident was used to solve a problem, whereas in other cases, like the UK riots, social media like Twitter and Facebook made situations worse by organizing mobs and spreading messages such as “let’s go to this street.”",
     "उदाहरण के लिए, 2009 की हडसन नदी की घटना का इस्तेमाल एक समस्या को हल करने के लिए किया गया था, जबकि अन्य मामलों में, जैसे कि यूके में हुए दंगों में, ट्विटर और फेसबुक जैसे सोशल मीडिया ने भीड़ को संगठित करके और \"चलो इस सड़क पर चलते हैं\" जैसे संदेश फैलाकर स्थिति को और खराब कर दिया।"),

    ("In another example, news articles talked about “Nepal earthquake: Government using social media to connect and provide relief.” Particularly in India, there is much more usage of social media for interacting with citizens.",
     "एक अन्य उदाहरण में, समाचार लेखों में \"नेपाल भूकंप: सरकार राहत पहुंचाने और लोगों से जुड़ने के लिए सोशल मीडिया का उपयोग कर रही है\" शीर्षक से चर्चा की गई। विशेष रूप से भारत में, नागरिकों से संवाद करने के लिए सोशल मीडिया का उपयोग बहुत अधिक होता है।"),

    ("Over the last year or year and a half, social networks have proliferated, and the government has been using these networks to organize themselves and interact with citizens.",
     "पिछले एक या डेढ़ साल में सोशल नेटवर्क का तेजी से विस्तार हुआ है, और सरकार इन नेटवर्कों का उपयोग खुद को संगठित करने और नागरिकों के साथ संवाद करने के लिए कर रही है।"),

    ("Another phenomenal example in the world was during revolutions where Twitter was used to connect with citizens and organize movements, which went viral through services like Twitter and Facebook.",
     "दुनिया में एक और अभूतपूर्व उदाहरण क्रांतियों के दौरान देखने को मिला, जब ट्विटर का इस्तेमाल नागरिकों से जुड़ने और आंदोलनों को संगठित करने के लिए किया गया, जो ट्विटर और फेसबुक जैसी सेवाओं के माध्यम से तेजी से फैल गए।"),

    ("So, in the UK riots, people were organizing themselves through BlackBerry Messenger and Facebook, which actually helped in making the riots worse.",
     "तो, UK दंगों में लोग ब्लैकबेरी मैसेंजर और फेसबुक के ज़रिए खुद को ऑर्गनाइज़ कर रहे थे, जिससे असल में दंगे और ज़्यादा बिगड़ गए।"),

    ("But at the same time, there were also positive uses of social media, where people created groups on Facebook and Twitter to actually help clean up after the riots.",
     "लेकिन साथ ही, सोशल मीडिया के कुछ पॉजिटिव इस्तेमाल भी हुए, जहाँ लोगों ने दंगों के बाद सफाई में मदद करने के लिए Facebook और Twitter पर ग्रुप बनाए।"),

    ("Here is another example: In India, when there was an earthquake, people started spreading rumors on social media that there was another earthquake coming.",
     "यहाँ एक और उदाहरण है: भारत में जब भूकंप आया, तो लोगों ने सोशल मीडिया पर अफवाहें फैलानी शुरू कर दीं कि एक और भूकंप आने वाला है।"),

    ("So this is another example of how social media, particularly platforms like Twitter and Facebook, can amplify fear and spread misinformation quickly.",
     "तो यह एक और उदाहरण है कि कैसे सोशल मीडिया, खासकर ट्विटर और फेसबुक जैसे प्लेटफॉर्म, डर को बढ़ा सकते हैं और गलत जानकारी को तेज़ी से फैला सकते हैं।"),

    ("One more example is the Mumbai attacks in 2008.",
     "इसका एक और उदाहरण 2008 में हुए मुंबई हमले हैं।"),

    ("Terrorists were actually following news channels and social media like Twitter to understand what the government and police were planning.",
     "आतंकवादी असल में यह समझने के लिए न्यूज़ चैनल और ट्विटर जैसे सोशल मीडिया को फॉलो कर रहे थे कि सरकार और पुलिस क्या प्लान बना रही हैं।"),

    ("This shows that social media can also be misused in serious crises",
     "इससे पता चलता है कि गंभीर संकटों में सोशल मीडिया का भी गलत इस्तेमाल किया जा सकता है।"),

    ("At the same time, during natural disasters like floods or earthquakes, we have also seen how social media helps people to share helpline numbers, provide shelter information, or connect missing people with their families.",
     "साथ ही, बाढ़ या भूकंप जैसी प्राकृतिक आपदाओं के दौरान, हमने यह भी देखा है कि सोशल मीडिया लोगों को हेल्पलाइन नंबर शेयर करने, शेल्टर की जानकारी देने या लापता लोगों को उनके परिवारों से जोड़ने में कैसे मदद करता है।"),

    ("For example, during the Chennai floods in 2015, Facebook and Twitter were widely used to organize rescue and relief efforts.",
     "उदाहरण के लिए, 2015 में चेन्नई बाढ़ के दौरान, बचाव और राहत कामों को ऑर्गनाइज़ करने के लिए फेसबुक और ट्विटर का बड़े पैमाने पर इस्तेमाल किया गया था।"),

    ("So, the point is, social media is just a tool.",
     "तो, बात यह है कि सोशल मीडिया सिर्फ एक टूल है।"),

    ("It can be used for positive purposes such as connecting people, saving lives, and crisis management, or for negative purposes such as spreading rumors, fear, and even helping criminals.",
     "इसका इस्तेमाल अच्छे कामों के लिए किया जा सकता है, जैसे लोगों को जोड़ना, जान बचाना और संकट प्रबंधन, या बुरे कामों के लिए, जैसे अफवाहें फैलाना, डर फैलाना और यहां तक ​​कि अपराधियों की मदद करना।"),

    ("Therefore, it is very important to understand both sides of social media.",
     "इसलिए, सोशल मीडिया के दोनों पहलुओं को समझना बहुत ज़रूरी है।"),

    ("Until now in the course, we have seen some logistics about the course — what online social media is, its impact and numbers, some ways in which we can actually use these social media services, and some examples of these social media services.",
     "अब तक इस कोर्स में, हमने कोर्स के बारे में कुछ लॉजिस्टिक्स देखे हैं - ऑनलाइन सोशल मीडिया क्या है, इसका असर और नंबर्स, कुछ तरीके जिनसे हम असल में इन सोशल मीडिया सर्विसेज़ का इस्तेमाल कर सकते हैं, और इन सोशल मीडिया सर्विसेज़ के कुछ उदाहरण।"),

    ("What I will cover today is actually looking at some of the incidences, both positive and negative, where social media has played a role.",
     "आज मैं जिन चीज़ों पर बात करूँगा, उनमें असल में कुछ ऐसी घटनाओं को देखेंगे, जो पॉजिटिव और नेगेटिव दोनों हैं, जिनमें सोशल मीडिया ने भूमिका निभाई है।"),

    ("Here is a first one: as I have put in the slide, it happened in 2009.",
     "यह पहला उदाहरण है: जैसा कि मैंने स्लाइड में दिखाया है, यह घटना 2009 में हुई थी।"),

    ("This is the first time ever a social media service like Twitter was actually used for crisis management.",
     "यह पहली बार है जब ट्विटर जैसी सोशल मीडिया सेवा का उपयोग वास्तव में संकट प्रबंधन के लिए किया गया है।"),

    ("Here is a tweet which jkrums actually posted, which reads: “There is a plane in the Hudson.",
     "यह वह ट्वीट है जिसे जेक्रुम्स ने वास्तव में पोस्ट किया था, जिसमें लिखा है: \"हडसन में एक विमान है।\""),

    ("I am on the ferry going to pick up the people.",
     "मैं लोगों को लेने के लिए नौका पर जा रहा हूँ।"),

    ("Crazy.” Until then, Twitter was basically used for conversations, like saying what I am doing in life in the morning — “Morning Monday”, “I am having coffee”, “I am traveling here,” and things like that.",
     "अजीब बात है। तब तक, ट्विटर का इस्तेमाल मुख्य रूप से बातचीत के लिए किया जाता था, जैसे कि सुबह मैं अपनी जिंदगी में क्या कर रहा हूँ, यह बताना - \"सुप्रभात सोमवार\", \"मैं कॉफी पी रहा हूँ\", \"मैं यहाँ यात्रा कर रहा हूँ,\" और इसी तरह की बातें।"),

    ("Whereas, for the first time in 2009, jkrums actually posted this tweet when the US Airways flight landed in the Hudson River.",
     "जबकि, 2009 में पहली बार, जेकेरम्स ने वास्तव में यह ट्वीट तब पोस्ट किया था जब यूएस एयरवेज की फ्लाइट हडसन नदी में उतरी थी।"),

    ("Before the first responders could reach, there were actually citizens who were helping in the situation.",
     "आपातकालीन सेवाओं के पहुंचने से पहले ही, कुछ नागरिक स्थिति को संभालने में मदद कर रहे थे।"),

    ("Therefore, Twitter and social media services started being used in many different ways.",
     "इसलिए, ट्विटर और सोशल मीडिया सेवाओं का उपयोग कई अलग-अलग तरीकों से किया जाने लगा।"),

    ("I am going to talk to you about some examples where social media played both positive and negative roles.",
     "मैं आपको कुछ ऐसे उदाहरणों के बारे में बताने जा रहा हूँ जहाँ सोशल मीडिया ने सकारात्मक और नकारात्मक दोनों भूमिकाएँ निभाई हैं।"),

    ("In this case, it is very positive because it helps in actually solving a crisis and helps first responders.",
     "इस मामले में, यह बहुत सकारात्मक है क्योंकि यह वास्तव में संकट को हल करने में मदद करता है और आपातकालीन सेवाओं में सहायता प्रदान करता है।"),

    ("Here is an example where in India there was a kid who was actually lost in a railway station, and somebody took a picture of this kid with a railway police officer.",
     "यहां एक उदाहरण दिया गया है जिसमें भारत में एक बच्चा रेलवे स्टेशन पर खो गया था, और किसी ने उस बच्चे की एक रेलवे पुलिस अधिकारी के साथ तस्वीर खींची।"),

    ("In 20 minutes, she was actually able to connect with her parents.",
     "सिर्फ 20 मिनट में वह अपने माता-पिता से संपर्क करने में सफल हो गई।"),

    ("The primary way by which it was done was posting the picture on social media, particularly tagging, mentioning the concerned people.",
     "इसे अंजाम देने का मुख्य तरीका सोशल मीडिया पर तस्वीर पोस्ट करना था, खासकर संबंधित लोगों को टैग करना और उनका जिक्र करना।"),

    ("Therefore, this tweet got viral, and the kid was able to connect with her parents within 20 minutes.",
     "इसलिए, यह ट्वीट वायरल हो गया और बच्ची 20 मिनट के भीतर अपने माता-पिता से संपर्क करने में सक्षम हो गई।"),

    ("Keeping some of these examples, particularly with missing children, there has also been organizations which have been started only to help finding missing children and parents through social media.",
     "इन उदाहरणों में से कुछ को ध्यान में रखते हुए, विशेष रूप से लापता बच्चों के मामलों में, ऐसे संगठन भी शुरू किए गए हैं जो केवल सोशल मीडिया के माध्यम से लापता बच्चों और माता-पिता को खोजने में मदद करने के लिए हैं।"),

    ("One example is \"Find Your Missing Child,\" which uses only social media services and provides tips on how one can connect with children to find missing children.",
     "इसका एक उदाहरण \"अपने लापता बच्चे को ढूंढें\" है, जो केवल सोशल मीडिया सेवाओं का उपयोग करता है और लापता बच्चों को खोजने के लिए बच्चों से जुड़ने के तरीके के बारे में सुझाव प्रदान करता है।"),

    ("Now, here is another example where a teenager was found dead just days after posting a message on social media.",
     "अब, यहां एक और उदाहरण है जहां एक किशोर सोशल मीडिया पर संदेश पोस्ट करने के कुछ ही दिनों बाद मृत पाया गया।"),

    ("This is a negative example, where the girl posted information on her Facebook account about being bullied.",
     "यह एक नकारात्मक उदाहरण है, जिसमें लड़की ने अपने फेसबुक अकाउंट पर अपने साथ हो रही बदमाशी के बारे में जानकारी पोस्ट की।"),

    ("Cyberbullying is a big problem on social media.",
     "साइबरबुलिंग सोशल मीडिया पर एक बड़ी समस्या है।"),

    ("Because of cyberbullying, people have actually killed themselves, and many times they leave a message on their own social networks.",
     "साइबरबुलिंग की वजह से लोग आत्महत्या भी कर लेते हैं, और कई बार वे अपने सोशल नेटवर्क पर संदेश छोड़ देते हैं।"),

    ("Now, we will look at the overview of Online Social Media.",
     "अब, हम ऑनलाइन सोशल मीडिया का ओवरव्यू देखेंगे।"),

    ("Social media is of different types, and different types of content are getting generated in our social media.",
     "सोशल मीडिया कई तरह का होता है, और हमारे सोशल मीडिया पर अलग-अलग तरह का कंटेंट जेनरेट हो रहा है।"),

    ("There are also many different ways in which social media content is generated.",
     "सोशल मीडिया कंटेंट बनाने के भी कई अलग-अलग तरीके हैं।"),

    ("For example, publishing platforms like Wikipedia and other crowd-sourced ways of creating content.",
     "उदाहरण के लिए, विकिपीडिया जैसे पब्लिशिंग प्लेटफॉर्म और कंटेंट बनाने के दूसरे क्राउड-सोर्स्ड तरीके।"),

    ("In addition, there are social games, virtual worlds, and other services through which content is being generated.",
     "इसके अलावा, सोशल गेम्स, वर्चुअल वर्ल्ड और दूसरी सर्विस भी हैं जिनके ज़रिए कंटेंट बनाया जा रहा है।"),

    ("Instagram is focused on images, while Twitter is a micro-blog platform with short posts but also supports multiple content types.",
     "इंस्टाग्राम इमेज पर फोकस करता है, जबकि ट्विटर एक माइक्रो-ब्लॉग प्लेटफॉर्म है जिसमें छोटे पोस्ट होते हैं, लेकिन यह कई तरह के कंटेंट को भी सपोर्ट करता है।"),

    ("For example, Foursquare is focused entirely on location, LinkedIn on professional connections, and so on. ",
     "उदाहरण के लिए, Foursquare पूरी तरह से लोकेशन पर फोकस करता है, LinkedIn प्रोफेशनल कनेक्शन पर, और इसी तरह।"),

    ("Recently, newer networks like Pinterest, Vine, Tumblr, Tinder, Whisper, Snapchat, and WeChat have become popular.",
     "हाल ही में, Pinterest, Vine, Tumblr, Tinder, Whisper, Snapchat और WeChat जैसे नए नेटवर्क पॉपुलर हुए हैं।"),

    ("Some of these are “ephemeral social networks,” where content disappears after a short time (e.g., Snapchat).",
     "इनमें से कुछ “कुछ समय के लिए चलने वाले सोशल नेटवर्क” हैं, जहाँ कंटेंट थोड़े समय के बाद गायब हो जाता है (जैसे, स्नैपचैट)।"),

    ("This shows how diverse social networks are becoming.",
     "यह दिखाता है कि सोशल नेटवर्क कितने विविध होते जा रहे हैं।"),

    ("Currently, there are around 215–220 popular social networks in existence, though only a handful are highlighted here to show the variety of content they support.",
     "अभी, लगभग 215–220 पॉपुलर सोशल नेटवर्क मौजूद हैं, हालांकि यहाँ सिर्फ़ कुछ ही नेटवर्क को दिखाया गया है ताकि यह पता चल सके कि वे किस तरह के कंटेंट को सपोर्ट करते हैं।"),

    ("Even in this course, there are around 1,500 people interested, largely due to the proliferation of online social networks and mobile phones.",
     "इस कोर्स में भी, लगभग 1,500 लोग इंटरेस्टेड हैं, जिसका मुख्य कारण ऑनलाइन सोशल नेटवर्क और मोबाइल फोन का बढ़ना है।"),

    ("In India alone, earlier it was said that 1 in every 13 people accessed social networks, and now it is closer to 1 in every 9 or 10.",
     "सिर्फ़ भारत में ही, पहले कहा जाता था कि हर 13 में से 1 व्यक्ति सोशल नेटवर्क इस्तेमाल करता था, और अब यह आंकड़ा हर 9 या 10 में से 1 व्यक्ति के करीब है।"),

    ("This rapid growth is driving massive amounts of content creation on social media.",
     "यह तेज़ ग्रोथ सोशल मीडिया पर बड़े पैमाने पर कंटेंट क्रिएशन को बढ़ावा दे रही है।"),

    ("Consider what happens online in just 60 seconds.",
     "सोचिए कि सिर्फ 60 सेकंड में ऑनलाइन क्या होता है।"),

    ("In 2013, Facebook had 2.5 million posts every 60 seconds, but now it has 3.5 million.",
     "2013 में, फेसबुक पर हर 60 सेकंड में 2.5 मिलियन पोस्ट होते थे, लेकिन अब यह संख्या 3.5 मिलियन हो गई है।"),

    ("Twitter had 278,000 posts every 60 seconds in 2013, and today it is around 420,000.",
     "2013 में ट्विटर पर हर 60 सेकंड में 278,000 पोस्ट होते थे, और आज यह संख्या लगभग 420,000 है।"),

    ("YouTube had about 100 hours of video uploaded every 60 seconds in 2013, but now it is 400 hours.",
     "2013 में YouTube पर हर 60 सेकंड में लगभग 100 घंटे के वीडियो अपलोड होते थे, लेकिन अब यह 400 घंटे हो गया है।"),

    ("This illustrates how much information is being uploaded continuously - Google searches, Instagram photos, WordPress posts, WhatsApp messages, emails, and more.",
     "यह दिखाता है कि कितनी सारी जानकारी लगातार अपलोड हो रही है - गूगल सर्च, इंस्टाग्राम फ़ोटो, वर्डप्रेस पोस्ट, WhatsApp मैसेज, ईमेल, और भी बहुत कुछ।"),

    ("The second is Variety—the different types of content across platforms.",
     "दूसरा है वैरायटी—अलग-अलग प्लेटफॉर्म पर अलग-अलग तरह का कंटेंट।"),

    ("Now let’s look at some popular social networks, starting with Facebook.",
     "अब आइए कुछ पॉपुलर सोशल नेटवर्क्स को देखते हैं, जिसकी शुरुआत फेसबुक से करते हैं।"),

    ("Many of you may already have an account.",
     "आपमें से कई लोगों का पहले से ही अकाउंट हो सकता है।"),

    ("The building blocks of Facebook include the feed (posts from friends), posts (which can be text, image, or video), likes, comments, and shares.",
     "फेसबुक के बिल्डिंग ब्लॉक्स में फीड (दोस्तों की पोस्ट), पोस्ट (जो टेक्स्ट, इमेज या वीडियो हो सकती हैं), लाइक, कमेंट और शेयर शामिल हैं।"),

    ("All of this content is stored in a graph structure—users and content as nodes, and edges representing friendships.",
     "यह सारा कंटेंट एक ग्राफ़ स्ट्रक्चर में स्टोर होता है - यूज़र्स और कंटेंट नोड्स के रूप में, और एज दोस्ती को दिखाते हैं।"),

    ("Facebook is a bidirectional network - both people must accept the connection to become friends.",
     "फेसबुक एक बाईडायरेक्शनल नेटवर्क है - दोस्त बनने के लिए दोनों लोगों को कनेक्शन स्वीकार करना होता है।"),

    ("Apart from this, there are pages (which you like or join) and groups (public or private) that allow communities to form.",
     "इसके अलावा, ऐसे पेज (जिन्हें आप लाइक या जॉइन करते हैं) और ग्रुप (पब्लिक या प्राइवेट) भी हैं जो कम्युनिटी बनाने की इजाज़त देते हैं।"),

    ("You can follow anyone, and they don’t need to follow you back.",
     "आप किसी को भी फॉलो कर सकते हैं, और उन्हें आपको वापस फॉलो करने की ज़रूरत नहीं है।"),

    ("Twitter is a micro-blogging site, limited to 140 characters (earlier).",
     "ट्विटर एक माइक्रो-ब्लॉगिंग साइट है, जिसमें 140 कैरेक्टर (पहले) की लिमिट थी।"),

    ("It includes followers and followings, tweets, retweets, replies, likes, and hashtags.",
     "इसमें फॉलोअर्स और फॉलोइंग, ट्वीट्स, रीट्वीट्स, रिप्लाई, लाइक्स और हैशटैग शामिल हैं।"),

    ("For example, hashtags like #PSOSMNPTEL can be used by the class to collect data.",
     "उदाहरण के लिए, क्लास डेटा इकट्ठा करने के लिए #PSOSMNPTEL जैसे हैशटैग का इस्तेमाल कर सकती है।"),

    ("LinkedIn is a professional network built on connections rather than friends or followers.",
     "लिंक्डइन एक प्रोफेशनल नेटवर्क है जो दोस्तों या फॉलोअर्स के बजाय कनेक्शन्स पर बना है।"),

    ("The focus is on jobs, recruitment, and professional updates.",
     "फोकस नौकरियों, भर्ती और प्रोफेशनल अपडेट पर है।"),
    ("Now, we will look at the overview of Online Social Media.", "अब, हम ऑनलाइन सोशल मीडिया का ओवरव्यू देखेंगे।"),
    ("Social media is of different types, and different types of content are getting generated in our social media.", "सोशल मीडिया कई तरह का होता है, और हमारे सोशल मीडिया पर अलग-अलग तरह का कंटेंट जेनरेट हो रहा है।"),
    ("There are also many different ways in which social media content is generated.", "सोशल मीडिया कंटेंट बनाने के भी कई अलग-अलग तरीके हैं।"),
    ("For example, publishing platforms like Wikipedia and other crowd-sourced ways of creating content.", "उदाहरण के लिए, विकिपीडिया जैसे पब्लिशिंग प्लेटफॉर्म और कंटेंट बनाने के दूसरे क्राउड-सोर्स्ड तरीके।"),
    ("In addition, there are social games, virtual worlds, and other services through which content is being generated.", "इसके अलावा, सोशल गेम्स, वर्चुअल वर्ल्ड और दूसरी सर्विस भी हैं जिनके ज़रिए कंटेंट बनाया जा रहा है।"),
    ("Instagram is focused on images, while Twitter is a micro-blog platform with short posts but also supports multiple content types.", "इंस्टाग्राम इमेज पर फोकस करता है, जबकि ट्विटर एक माइक्रो-ब्लॉग प्लेटफॉर्म है जिसमें छोटे पोस्ट होते हैं, लेकिन यह कई तरह के कंटेंट को भी सपोर्ट करता है।"),
    ("For example, Foursquare is focused entirely on location, LinkedIn on professional connections, and so on.", "उदाहरण के लिए, Foursquare पूरी तरह से लोकेशन पर फोकस करता है, LinkedIn प्रोफेशनल कनेक्शन पर, और इसी तरह।"),
    ("Recently, newer networks like Pinterest, Vine, Tumblr, Tinder, Whisper, Snapchat, and WeChat have become popular.", "हाल ही में, Pinterest, Vine, Tumblr, Tinder, Whisper, Snapchat और WeChat जैसे नए नेटवर्क पॉपुलर हुए हैं।"),
    ("Some of these are “ephemeral social networks,” where content disappears after a short time (e.g., Snapchat).", "इनमें से कुछ “कुछ समय के लिए चलने वाले सोशल नेटवर्क” हैं, जहाँ कंटेंट थोड़े समय के बाद गायब हो जाता है (जैसे, स्नैपचैट)।"),
    ("This shows how diverse social networks are becoming.", "यह दिखाता है कि सोशल नेटवर्क कितने विविध होते जा रहे हैं।"),
    ("Currently, there are around 215–220 popular social networks in existence, though only a handful are highlighted here to show the variety of content they support.", "अभी, लगभग 215–220 पॉपुलर सोशल नेटवर्क मौजूद हैं, हालांकि यहाँ सिर्फ़ कुछ ही नेटवर्क को दिखाया गया है ताकि यह पता चल सके कि वे किस तरह के कंटेंट को सपोर्ट करते हैं।"),
    ("Even in this course, there are around 1,500 people interested, largely due to the proliferation of online social networks and mobile phones.", "इस कोर्स में भी, लगभग 1,500 लोग इंटरेस्टेड हैं, जिसका मुख्य कारण ऑनलाइन सोशल नेटवर्क और मोबाइल फोन का बढ़ना है।"),
    ("In India alone, earlier it was said that 1 in every 13 people accessed social networks, and now it is closer to 1 in every 9 or 10.", "सिर्फ़ भारत में ही, पहले कहा जाता था कि हर 13 में से 1 व्यक्ति सोशल नेटवर्क इस्तेमाल करता था, और अब यह आंकड़ा हर 9 या 10 में से 1 व्यक्ति के करीब है।"),
    ("This rapid growth is driving massive amounts of content creation on social media.", "यह तेज़ ग्रोथ सोशल मीडिया पर बड़े पैमाने पर कंटेंट क्रिएशन को बढ़ावा दे रही है।"),
    ("Consider what happens online in just 60 seconds.", "सोचिए कि सिर्फ 60 सेकंड में ऑनलाइन क्या होता है।"),
    ("In 2013, Facebook had 2.5 million posts every 60 seconds, but now it has 3.5 million.", "2013 में, फेसबुक पर हर 60 सेकंड में 2.5 मिलियन पोस्ट होते थे, लेकिन अब यह संख्या 3.5 मिलियन हो गई है।"),
    ("Twitter had 278,000 posts every 60 seconds in 2013, and today it is around 420,000.", "2013 में ट्विटर पर हर 60 सेकंड में 278,000 पोस्ट होते थे, और आज यह संख्या लगभग 420,000 है।"),
    ("YouTube had about 100 hours of video uploaded every 60 seconds in 2013, but now it is 400 hours.", "2013 में YouTube पर हर 60 सेकंड में लगभग 100 घंटे के वीडियो अपलोड होते थे, लेकिन अब यह 400 घंटे हो गया है।"),
    ("This illustrates how much information is being uploaded continuously - Google searches, Instagram photos, WordPress posts, WhatsApp messages, emails, and more.", "यह दिखाता है कि कितनी सारी जानकारी लगातार अपलोड हो रही है - गूगल सर्च, इंस्टाग्राम फ़ोटो, वर्डप्रेस पोस्ट, WhatsApp मैसेज, ईमेल, और भी बहुत कुछ।"),
    ("The second is Variety—the different types of content across platforms.", "दूसरा है वैरायटी—अलग-अलग प्लेटफॉर्म पर अलग-अलग तरह का कंटेंट।"),
    ("Now let’s look at some popular social networks, starting with Facebook.", "अब आइए कुछ पॉपुलर सोशल नेटवर्क्स को देखते हैं, जिसकी शुरुआत फेसबुक से करते हैं।"),
    ("Many of you may already have an account.", "आपमें से कई लोगों का पहले से ही अकाउंट हो सकता है।"),
    ("The building blocks of Facebook include the feed (posts from friends), posts (which can be text, image, or video), likes, comments, and shares.", "फेसबुक के बिल्डिंग ब्लॉक्स में फीड (दोस्तों की पोस्ट), पोस्ट (जो टेक्स्ट, इमेज या वीडियो हो सकती हैं), लाइक, कमेंट और शेयर शामिल हैं।"),
    ("All of this content is stored in a graph structure—users and content as nodes, and edges representing friendships.", "यह सारा कंटेंट एक ग्राफ़ स्ट्रक्चर में स्टोर होता है - यूज़र्स और कंटेंट नोड्स के रूप में, और एज दोस्ती को दिखाते हैं।"),
    ("Facebook is a bidirectional network - both people must accept the connection to become friends.", "फेसबुक एक बाईडायरेक्शनल नेटवर्क है - दोस्त बनने के लिए दोनों लोगों को कनेक्शन स्वीकार करना होता है।"),
    ("Apart from this, there are pages (which you like or join) and groups (public or private) that allow communities to form.", "इसके अलावा, ऐसे पेज (जिन्हें आप लाइक या जॉइन करते हैं) और ग्रुप (पब्लिक या प्राइवेट) भी हैं जो कम्युनिटी बनाने की इजाज़त देते हैं।"),
    ("You can follow anyone, and they don’t need to follow you back.", "आप किसी को भी फॉलो कर सकते हैं, और उन्हें आपको वापस फॉलो करने की ज़रूरत नहीं है।"),
    ("Twitter is a micro-blogging site, limited to 140 characters (earlier).", "ट्विटर एक माइक्रो-ब्लॉगिंग साइट है, जिसमें 140 कैरेक्टर (पहले) की लिमिट थी।"),
    ("It includes followers and followings, tweets, retweets, replies, likes, and hashtags.", "इसमें फॉलोअर्स और फॉलोइंग, ट्वीट्स, रीट्वीट्स, रिप्लाई, लाइक्स और हैशटैग शामिल हैं।"),
    ("For example, hashtags like #PSOSMNPTEL can be used by the class to collect data.", "उदाहरण के लिए, क्लास डेटा इकट्ठा करने के लिए #PSOSMNPTEL जैसे हैशटैग का इस्तेमाल कर सकती है।"),
    ("LinkedIn is a professional network built on connections rather than friends or followers.", "लिंक्डइन एक प्रोफेशनल नेटवर्क है जो दोस्तों या फॉलोअर्स के बजाय कनेक्शन्स पर बना है।"),
    ("The focus is on jobs, recruitment, and professional updates.", "फोकस नौकरियों, भर्ती और प्रोफेशनल अपडेट पर है।")
]

gold_hi = [
    "1-3 3-2 4-4 5-8 6-9 7-12 9-11 11-5 13-15 14-16 15-17 18-25 19-22 21-20 23-30",
    "0-2 1-0 2-4 3-3 5-6 6-8 7-20 8-10 9-17 10-14 11-11 12-12 13-21 14-28 15-22 16-23 17-25 19-27 21-36 22-37 24-32 25-33 26-34 28-46 29-49 31-41 32-39 34-45 35-44 38-52 39-53",
    "0-6 1-1 2-2 3-4 4-5 5-27 7-7 8-8 9-9 11-18 12-19 13-17 14-15 15-12 17-10 18-30 19-34 20-33 22-48 23-45 24-46 25-21 26-20 29-40 30-37 31-36 32-35",
    "2-0 3-4 4-2 8-3 9-6 10-7 12-11 13-14 15-15 18-19 19-16 20-17 21-31 22-22 23-20 25-28 26-27 27-25",
    "0-2 1-4 2-5 3-1 5-0 7-8 8-6 9-12 10-13 12-15 13-25 14-18 15-17 16-16 17-19 19-20 20-28 23-35 24-33 25-32 28-31",
    "0-0 1-3 1-19 2-3 2-19 4-2 4-20 5-4 6-16 7-5 7-13 8-11 12-7 12-21 13-8 13-16 14-17 15-18 17-3 17-19 19-3 19-19 20-2 20-20 21-23 21-24",
    "0-0 4-2 6-10 7-9 8-7 9-8 10-5 10-15 10-21 11-3 12-4 12-10 13-11 14-12 15-28 16-27 17-26 18-23 19-24 20-25 21-22 23-19 24-17 26-16 28-14 28-28",
    "0-0 1-4 2-1 2-22 5-5 6-7 8-9 8-27 10-8 10-9 10-24 11-11 12-18 13-17 14-16 15-15 16-13 17-14 18-21 20-9 20-27 21-1 21-22 22-8 22-9 22-24 23-25 23-27",
    "0-0 1-1 2-5 3-2 4-4 6-7 7-8 8-9 9-10 10-15 12-11 13-3 13-12 13-21 14-13 14-15 15-19 15-28 16-18 17-16 18-3 18-12 18-21 19-27 20-22 20-23 21-25 21-29",
    "0-1 1-2 2-3 3-9 5-7 6-8 7-5 8-4",
    "0-0 1-18 1-27 3-15 4-7 5-8 6-9 6-21 7-12 8-13 9-11 10-10 12-4 13-23 15-20 16-9 16-21 17-22 18-18 18-27 19-24 19-27",
    "0-0 1-2 2-4 3-8 4-9 5-16 6-11 8-12 8-13 9-7 10-5 11-6",
    "0-0 3-1 4-9 5-6 6-7 7-5 8-2 9-3 9-28 10-9 11-10 13-12 14-13 15-37 16-16 17-17 18-38 19-18 19-30 21-22 22-20 23-21 23-23 24-27 25-20 26-26 27-3 27-28 28-35 30-18 30-30 31-34 32-32 33-33 33-40",
    "0-2 0-17 1-0 1-2 1-17 2-8 4-5 5-6 7-3 8-18 9-10 9-19 10-20 11-28 12-22 12-24 13-25 14-2 14-17 15-15 17-10 17-19 18-11 19-12 19-28",
    "0-0 2-1 3-3 3-10 4-5 5-6 6-3 6-10 7-7 8-8 9-10",
    "0-0 1-8 2-7 3-1 4-5 4-23 5-2 6-3 6-21 7-10 7-24 9-13 10-11 10-13 11-15 12-14 13-16 13-29 14-17 15-18 16-19 17-5 17-23 18-20 19-3 19-21 20-10 20-24 22-26 22-28 23-25 23-26 23-28 24-27 25-16 25-29 26-30 27-35 28-33 28-36",
    "0-0 2-10 3-8 4-9 6-7 7-4 8-5 10-1 11-2 11-10",
    "0-1 1-0 2-4 2-9 2-29 4-3 4-4 4-6 4-9 4-29 5-5 5-27 7-12 8-10 8-24 8-45 9-11 10-8 12-3 12-4 12-6 12-9 12-29 13-14 14-18 15-15 16-16 16-31 16-41 17-17 17-32 17-42 18-19 19-20 20-21 21-22 21-39 22-23 23-10 23-24 23-45 24-25 25-4 25-9 25-29 26-26 27-5 27-27 28-37 29-28 30-35 31-30 31-40 32-16 32-31 32-41 33-17 33-32 33-42 34-13 34-33 34-38 34-43 34-46 35-22 35-39 36-10 36-24 36-45 37-46 39-30 39-40 40-16 40-31 40-41 41-17 41-32 41-42 42-13 42-33 42-38 42-43 42-46",
    "1-1 2-6 3-5 4-0 6-8 7-14 9-10 11-9 12-12 13-19 15-17 16-14 16-18 17-21 18-22 19-23 20-24 21-26 23-25 23-27",
    "0-0 0-11 1-3 1-10 3-1 4-3 4-10 5-4 6-6 8-9 9-8 9-14 10-8 10-14 11-2 11-3 11-10 12-0 12-11 13-15 14-8 14-14 15-13",
    "0-0 1-3 3-1 6-10 7-7 8-8 9-9 10-6 11-5 12-19 13-12 14-11 15-17 16-14 17-15 17-20",
    "0-0 1-3 1-14 1-19 2-17 3-2 4-4 4-12 5-5 6-7 7-9 7-11 8-4 8-12 9-3 9-13 9-14 10-15 11-3 11-14 11-19 12-17 13-18 14-8 14-16 16-19",
    "0-0 1-10 2-7 4-6 5-8 6-5 7-3 10-1 10-10",
    "0-0 1-4 2-3 2-4 3-5 4-16 5-8 6-7 7-13 8-11 8-16 9-17 11-24 12-20 12-33 12-38 13-27 13-37 13-43 14-25 14-41 15-23 16-22 17-23 19-19 20-30 21-31 22-32 23-20 23-33 23-38 24-27 24-37 24-43 26-27 26-34 26-37 26-43 27-20 27-33 27-38 28-27 28-37 28-43 29-40 30-27 30-37 30-39 30-43 31-44 32-48 33-17 34-48",
    "0-0 3-3 4-4 5-2 5-8 5-22 6-1 7-5 8-7 9-12 10-9 11-10 12-15 14-16 15-17 16-19 17-23 18-2 18-8 18-22 21-21 21-24",
    "0-5 1-2 1-12 3-1 5-3 5-6 7-16 9-8 11-16 12-13 14-2 14-12 15-9 15-16",
    "0-0 1-1 2-2 3-3 4-4 5-5 6-14 7-13 8-7 9-11 10-8 11-9 12-10 12-14",
    "0-0 2-9 4-8 6-1 7-6 8-2 9-4 10-12 11-13 12-14 13-21 14-19 15-16 16-17 17-18 18-20 18-22",
    "1-0 2-1 2-2 2-10 2-15 2-22 3-3 3-8 4-6 4-18 5-4 6-5 7-7 8-3 8-8 9-16 11-9 12-13 12-14 14-11 15-19 16-16 17-20 18-21 18-26",
    "0-0 1-5 2-1 2-9 2-23 3-2 4-6 5-8 6-7 8-4 8-15 8-16 9-1 9-9 9-23 10-10 10-21 12-4 12-15 12-16 14-14 15-8 16-1 16-9 16-23 17-11 17-24 17-30 18-12 18-16 19-17 20-18 21-30 22-1 22-9 22-23 23-29 25-20 26-10 26-21 27-28 28-1 28-9 28-23 29-11 29-24 29-30 30-25 31-26 31-30",
    "0-3 0-10 1-1 2-2 3-4 6-11 8-8 9-7 10-5 11-6 11-13",
    "1-4 2-5 5-0 6-12 8-12 9-10 11-9 12-8 13-6 14-7 14-12 15-13 16-17 19-14 20-15 20-22",
    "0-0 1-1 2-2 3-5 4-3 5-6 7-7 9-18 11-15 13-12 14-13 15-11 16-8 17-9 17-20",
    "1-4 2-2 2-7 2-17 2-38 3-0 4-1 4-9 5-10 7-13 7-32 8-2 8-7 8-14 8-17 8-33 8-38 11-20 13-19 14-25 15-24 17-21 18-26 20-39 21-37 22-13 22-32 23-2 23-7 23-14 23-17 23-33 23-38 24-34 25-35 26-30 27-27 28-28 28-43",
    "0-1 1-2 2-8 2-17 3-3 3-7 3-22 4-3 5-4 5-19 6-5 6-7 6-8 6-17 7-9 8-8 8-15 8-17 9-10 10-11 11-12 12-13 13-18 14-34 15-33 18-1 20-27 21-26 22-20 22-25 22-36 24-3 24-7 24-22 25-4 25-19 26-20 26-25 26-36",
    "0-0 1-1 2-5 3-2 3-7 4-4 5-6 6-2 6-7 7-8 8-22 9-21 10-20 12-18 13-19 14-13 15-2 15-7 16-12 17-11 18-9 19-10 19-22",
    "0-0 1-4 2-1 3-2 4-3 4-4 5-5 7-6 8-21 9-20 10-11 11-8 11-12 12-9 13-10 14-18 15-16 16-16 16-22",
    "0-0 1-7 2-4 3-5 4-6 5-3 6-1 7-2 7-7",
    "0-2 1-3 2-0 3-4 6-5 7-9 8-10 9-11 10-12 11-13 12-19 14-18 15-17 16-14 18-15 19-16 19-21",
    "0-0 1-1 3-7 5-5 6-6 8-2 9-3 10-4",
    "0-0 1-1 4-2 5-3 6-7 7-12 8-13 9-14 10-15 13-16 15-8 16-9 17-10",
    "2-5 3-6 4-7 5-8 8-0 9-1 10-2 12-3",
    "1-0 2-5 3-6 4-4 5-3 6-7 7-11 8-12 9-13 11-9 12-8",
    "4-2 5-3 6-4 7-5 8-6 9-7 10-8 11-12 12-11 13-13 16-14",
    "0-0 2-3 3-2 4-1 5-6 6-7 8-8 9-9 10-10 12-13 13-14 14-17 15-24 16-25 17-19 18-22",
    "1-0 2-3 4-9 5-4 6-8 7-7 8-12 9-15 10-13 11-14 12-16 13-17 14-18",
    "1-12 2-13 3-11 4-3 5-4 6-5 7-6 8-7 9-8 10-9 11-10 13-15 14-14",
    "0-2 1-1 2-0 5-9 6-10 7-12 8-13 9-18 10-17 12-14 13-15 14-22 15-23",
    "0-0 1-1 2-6 3-7 4-4 5-5",
    "0-0 3-1 4-2 5-3 6-4 7-5 9-6 10-8 11-10 13-11 15-15 16-9 20-26 22-28 23-24 24-30",
    "0-3 1-2 2-0 3-1 6-4 7-5 8-6 9-7 11-11 14-19 16-12 17-13 18-14 19-15 20-16 21-17",
    "0-2 1-1 2-0 3-4 6-5 7-8 8-13 9-11 10-9 11-10 12-14 13-17 14-15 15-16 16-20 17-21 18-22 20-33 22-30 23-28 24-24 25-25 26-26 27-27",
    "0-0 1-1 2-2 8-9 9-10 10-5 11-3 12-4",
    "0-0 1-7 2-8 3-6 4-5 5-2 6-3 7-4",
    "0-1 1-0 2-2 4-8 5-9 6-10 7-4 8-5 9-6 10-13 11-14 12-15 14-17 15-18",
    "0-2 2-8 3-9 4-4 5-5 6-6 7-7 8-0 9-12 10-13 11-14 13-16 14-17",
    "0-2 3-9 4-10 6-12 7-7 8-4 9-5 12-0 13-16 14-17 16-23 18-20",
    "0-0 1-1 2-4 3-5 4-6 7-8 8-7 9-12 10-13 11-14 12-15 13-16 14-17 15-18 16-19 17-20 18-21 19-24",
    "1-0 3-2 4-3 5-7 6-8 7-9 9-4",
    "0-0 2-7 4-2 5-3 6-4 7-5 8-10 9-12 10-11",
    "0-2 2-0 4-5 7-8",
    "2-3 4-0 5-22 8-8 10-6 11-9 12-10 15-11 17-13 16-12 18-14 19-18 20-19 21-20 22-21",
    "0-1 2-0 3-2 5-7 6-6 7-3 8-4 9-10 10-11 11-12 13-13 14-17 15-18 16-21 17-19",
    "0-0 2-1 3-2 4-3 6-10 7-11 9-14 11-13 13-7 14-6",
    "5-3 6-4 7-5 8-6 9-7 10-8 11-11 12-12 13-13 14-14 15-15 16-18 17-22 18-19 20-20",
    "0-0 2-4 3-1 4-8 5-9 7-15 9-12 10-10 11-11",
    "0-0 2-1 3-2 4-3 5-10 7-6 8-7 9-8",
    "2-1 3-2 4-3 5-4 6-5 7-6 8-7 9-8 10-9",
    "1-0 2-11 3-10 4-9 7-13 10-3 12-5 13-4",
    "0-0 2-1 3-2 4-3 5-13 6-12 7-11 10-6 11-7 12-8",
    "1-0 3-6 4-1 5-2 6-3 7-4 8-5",
    "0-0 1-1 3-7 5-5 6-6 8-2 9-3 10-4",
    "0-0 1-1 4-2 5-3 6-7 7-12 8-13 9-14 10-15 13-16 15-8 16-9 17-10",
    "2-5 3-6 4-7 5-8 8-0 9-1 10-2 12-3",
    "1-0 2-5 3-6 4-4 5-3 6-7 7-11 8-12 9-13 11-9 12-8",
    "4-2 5-3 6-4 7-5 8-6 9-7 10-8 11-12 12-11 13-13 16-14",
    "0-0 2-3 3-2 4-1 5-6 6-7 8-8 9-9 10-10 12-13 13-14 14-17 15-24 16-25 17-19 18-22",
    "1-0 2-3 4-9 5-4 6-8 7-7 8-12 9-15 10-13 11-14 12-16 13-17 14-18",
    "1-12 2-13 3-11 4-3 5-4 6-5 7-6 8-7 9-8 10-9 11-10 13-15 14-14",
    "0-2 1-1 2-0 5-9 6-10 7-12 8-13 9-18 10-17 12-14 13-15 14-22 15-23",
    "0-0 1-1 2-6 3-7 4-4 5-5",
    "0-0 3-1 4-2 5-3 6-4 7-5 9-6 10-8 11-10 13-11 15-15 16-9 20-26 22-28 23-24 24-30",
    "0-3 1-2 2-0 3-1 6-4 7-5 8-6 9-7 11-11 14-19 16-12 17-13 18-14 19-15 20-16 21-17",
    "0-2 1-1 2-0 3-4 6-5 7-8 8-13 9-11 10-9 11-10 12-14 13-17 14-15 15-16 16-20 17-21 18-22 20-33 22-30 23-28 24-24 25-25 26-26 27-27",
    "0-0 1-1 2-2 8-9 9-10 10-5 11-3 12-4",
    "0-0 1-7 2-8 3-6 4-5 5-2 6-3 7-4",
    "0-1 1-0 2-2 4-8 5-9 6-10 7-4 8-5 9-6 10-13 11-14 12-15 14-17 15-18",
    "0-2 2-8 3-9 4-4 5-5 6-6 7-7 8-0 9-12 10-13 11-14 13-16 14-17",
    "0-2 3-9 4-10 6-12 7-7 8-4 9-5 12-0 13-16 14-17 16-23 18-20",
    "0-0 1-1 2-4 3-5 4-6 7-8 8-7 9-12 10-13 11-14 12-15 13-16 14-17 15-18 16-19 17-20 18-21 19-24",
    "1-0 3-2 4-3 5-7 6-8 7-9 9-4",
    "0-0 2-7 4-2 5-3 6-4 7-5 8-10 9-12 10-11",
    "0-2 2-0 4-5 7-8",
    "2-3 4-0 5-22 8-8 10-6 11-9 12-10 15-11 17-13 16-12 18-14 19-18 20-19 21-20 22-21",
    "0-1 2-0 3-2 5-7 6-6 7-3 8-4 9-10 10-11 11-12 13-13 14-17 15-18 16-21 17-19",
    "0-0 2-1 3-2 4-3 6-10 7-11 9-14 11-13 13-7 14-6",
    "5-3 6-4 7-5 8-6 9-7 10-8 11-11 12-12 13-13 14-14 15-15 16-18 17-22 18-19 20-20",
    "0-0 2-4 3-1 4-8 5-9 7-15 9-12 10-10 11-11",
    "0-0 2-1 3-2 4-3 5-10 7-6 8-7 9-8",
    "2-1 3-2 4-3 5-4 6-5 7-6 8-7 9-8 10-9",
    "1-0 2-11 3-10 4-9 7-13 10-3 12-5 13-4",
    "0-0 2-1 3-2 4-3 5-13 6-12 7-11 10-6 11-7 12-8",
    "1-0 3-6 4-1 5-2 6-3 7-4 8-5"
]

telugu_pairs = [
    ("I am going to the library.", "నేను గ్రంథాలయానికి వెళ్తున్నాను."),
    ("The weather is very beautiful today.", "ఈ రోజు వాతావరణం చాలా అందంగా ఉంది."),
    ("Can you please help me with this?", "దయచేసి దీనిలో నాకు సహాయం చేయగలరా?"),
    ("Where is the nearest hospital?", "అత్యంత సమీపంలోని ఆసుపత్రి ఎక్కడ ఉంది?"),
    ("I love eating biryani.", "నాకు బిర్యానీ తినడం చాలా ఇష్టం."),
    ("Reading books improves knowledge.", "పుస్తకాలు చదవడం జ్ఞానాన్ని పెంచుతుంది."),
    ("She is my best friend.", "ఆమె నా బెస్ట్ ఫ్రెండ్."),
    ("The train arrives at 5 PM.", "రైలు సాయంత్రం 5 గంటలకు వస్తుంది."),
    ("Please close the door.", "దయచేసి తలుపు మూసేయండి."),
    ("He works in a software company.", "అతను ఒక సాఫ్ట్‌వేర్ కంపెనీలో పని చేస్తాడు."),
    ("Hyderabad is a major IT hub.", "హైదరాబాద్ ఒక ప్రధాన ఐటి కేంద్రం."),
    ("Water boils at 100 degrees Celsius.", "నీరు 100 డిగ్రీల సెల్సియస్ వద్ద మరిగుతుంది."),
    ("Honesty is the best policy.", "నిజాయితీనే ఉత్తమ విధానం."),
    ("The sun rises in the east.", "సూర్యుడు తూర్పున ఉదయిస్తాడు."),
    ("I need to buy some vegetables.", "నాకు కొంత కూరగాయలు కొనాలి.")
]

# GOLD STANDARDS (Manual Sure Links)

gold_te = [
    "0-0 5-1 3-1 4-1 2-2 1-2",     # library+to the = గ్రంథాలయానికి
    "5-0 5-1 1-2 3-3 4-4 2-5",     # today = ఈ రోజు
    "2-0 6-1 5-1 4-2 3-3 0-4 1-4", # please=దయచేసి, me=నాకు, help=సహాయం
    "3-0 3-1 4-2 0-3 1-4",         # Where is the nearest hospital?
    "0-0 3-1 2-2 1-3 1-4",         # I love eating biryani.
    "1-0 0-1 3-2 2-3",             # Reading books improves knowledge.
    "0-0 2-1 3-2 4-3 1-3",         # She is my best friend.
    "1-0 5-1 4-2 3-3 2-4",         # train arrives at 5 PM.
    "0-0 3-1 1-2",                 # close the door.
    "0-0 4-2 5-3 2-3 1-4 1-5",     # works = పని చేస్తాడు
    "0-0 2-1 3-2 4-3 5-4",         # major IT hub.
    "0-0 3-1 4-2 5-3 2-4 1-5",     # boils = మరిగుతుంది
    "0-0 3-1 4-2",                 # Honesty is the best policy.
    "1-0 5-1 3-1 2-2",             # sun rises in the east.
    "0-0 4-1 5-2 1-3 3-3"          # need to buy some vegetables.
]



In [ ]:
def calculate_dynamic_score(gold_str, predicted_list, eng_sent):
    eng_tokens = eng_sent.lower().replace('.','').split()
    grammar_words = {'to', 'the', 'is', 'am', 'are', 'a', 'an', 'in', 'at', 'of', 'for', 'with'}

    sure_links = set()
    for p in gold_str.split():
        s, t = map(int, p.split('-'))
        sure_links.add((s, t))

    pred_links = set((i, j) for i, j, _ in predicted_list)

    missed_links = sure_links - pred_links
    recall_errors = 0
    for (s, t) in missed_links:
        word = eng_tokens[s] if s < len(eng_tokens) else ""
        if MIXER['STRICT_METRIC'] or (word not in grammar_words):
            recall_errors += 1

    extra_links = pred_links - sure_links
    precision_errors = 0
    for (s, t) in extra_links:
        if not any(abs(s-gs) <= 1 and abs(t-gt) <= 1 for (gs, gt) in sure_links):
            precision_errors += 0.25

    total_expected = len(sure_links)
    total_errors = recall_errors + (precision_errors * 0.3)

    if total_expected == 0: return 0.0
    return min(1.0, total_errors / total_expected)



In [ ]:
def run_pipeline(eng, tgt, lang_type="hi"):
    # Pre-processing: simple cleaning to improve alignment
    e_words = eng.replace("?", "").replace(".", "").split()
    t_words = tgt.replace("।", "").replace("?", "").replace(".", "").split()

    # 1. Similarity Matrix (LaBSE)
    e_emb = sim_model.encode(e_words, convert_to_tensor=True).cpu().numpy()
    t_emb = sim_model.encode(t_words, convert_to_tensor=True).cpu().numpy()
    sim_mat = cosine_similarity(e_emb, t_emb)

    # 2. Structural Alignment (IndicBERT v2 - Layer 8)
    inp = awesome_tokenizer([eng, tgt], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        out = awesome_model(**inp)

    layer_8 = out.hidden_states[8]
    src_tokens = awesome_tokenizer.tokenize(eng)
    tgt_tokens = awesome_tokenizer.tokenize(tgt)

    def map_subwords_indic(tokens):
        word_ids = []
        current = -1
        for tok in tokens:
            if tok.startswith(" "): current += 1
            elif current == -1: current = 0
            word_ids.append(current)
        return word_ids

    src_map, tgt_map = map_subwords_indic(src_tokens), map_subwords_indic(tgt_tokens)
    # Align subwords using symmetric softmax
    s_emb = layer_8[0, 1:len(src_tokens)+1]
    t_emb_feat = layer_8[1, 1:len(tgt_tokens)+1]
    dot = torch.matmul(s_emb, t_emb_feat.t())
    a_probs = ((torch.nn.functional.softmax(dot, dim=-1) + torch.nn.functional.softmax(dot, dim=-2)) / 2).cpu().numpy()

    # 3. CONSENSUS LOGIC (Intersection + High-Confidence Union)
    final_links = set()
    linked_src = set()
    linked_tgt = set()

    # Part A: Mutual Best Matches (High Precision)
    for i in range(len(e_words)):
        for j in range(len(t_words)):
            is_best_for_e = (j == np.argmax(sim_mat[i, :]))
            is_best_for_t = (i == np.argmax(sim_mat[:, j]))
            if is_best_for_e and is_best_for_t and sim_mat[i, j] > MIXER['SIMALIGN_THRESHOLD']:
                final_links.add((i, j, 1.0))
                linked_src.add(i); linked_tgt.add(j)

    # Part B: Add high-confidence structural links from IndicBERT
    for i in range(len(src_tokens)):
        for j in range(len(tgt_tokens)):
            if a_probs[i, j] > 0.4: # Very strong structural link
                wi, wj = src_map[i], tgt_map[j]
                if wi < len(e_words) and wj < len(t_words):
                    final_links.add((wi, wj, 0.8))
                    linked_src.add(wi); linked_tgt.add(wj)

    # Part C: Particle/Orphan Rescue
    ignore_hi = {'है', 'हैं', 'था', 'थी', 'के', 'को', 'में', 'से', 'ने', 'का', 'की', 'पर', 'और', 'तो', 'ही', 'भी', 'एक'}
    ignore_te = {'మరియు', 'లో', 'నుండి', 'యొక్క', 'ఉంది', 'ఉన్నారు'}
    stop_list = ignore_hi if lang_type == "hi" else ignore_te

    for i in range(len(e_words)):
        if i not in linked_src:
            best_j = np.argmax(sim_mat[i])
            if sim_mat[i][best_j] > MIXER['RESCUE_THRESHOLD']:
                if t_words[best_j] not in stop_list:
                    final_links.add((i, best_j, 0.5))

    return e_words, t_words, list(final_links)



In [ ]:
def render_output(src, tgt, aligns, idx, lang, score):
    # Modified for CLI: Print text summary instead of returning HTML
    print(f"#{idx} {lang} | Error: {score:.2f}")

    mapping_strs = []
    # Sort aligns by source index for readability
    sorted_aligns = sorted(aligns, key=lambda x: x[0])

    for i, j, _ in sorted_aligns:
        if i < len(src) and j < len(tgt):
            src_word = src[i]
            tgt_word = tgt[j]
            mapping_strs.append(f"{src_word} ↔ {tgt_word}")

    print("Mappings: " + " , ".join(mapping_strs))
    return ""

print("\n📊 RESULTS: HINDI")
m_avg = 0
for i, (eng, hindi) in enumerate(hindi_pairs):
    ew, mw, res = run_pipeline(eng, hindi, lang_type="hi")
    score = calculate_dynamic_score(gold_hi[i], res, eng)
    m_avg += score
    render_output(ew, mw, res, i+1, "Hindi", score)
print(f"🔴 Avg Hindi Error: {m_avg/len(hindi_pairs):.2f}")
with open("last_run_stats.txt", "w") as f:
    f.write(f"Hindi_Error: {m_avg/len(hindi_pairs):.2f}\n")

print("\n📊 RESULTS: TELUGU")
t_avg = 0
for i, (eng, telugu) in enumerate(telugu_pairs):
    ew, tw, res = run_pipeline(eng, telugu, lang_type="te")
    score = calculate_dynamic_score(gold_te[i], res, eng)
    t_avg += score
    render_output(ew, tw, res, i+1, "Telugu", score)
print(f"🔴 Avg Telugu Error: {t_avg/len(telugu_pairs):.2f}")


📊 RESULTS: HINDI
#1 Hindi | Error: 0.10
Mappings: Also, ↔ अलावा, , Also, ↔ इसके , in ↔ में , world, ↔ दुनिया , when ↔ जब , social ↔ सोशल , media ↔ मीडिया , started ↔ शुरू , being ↔ करने , used ↔ इस्तेमाल , for ↔ लिए , crises, ↔ संकटों , first ↔ पहली , time ↔ जब , it ↔ इसका , was ↔ थी , used ↔ इस्तेमाल , to ↔ को , propagate ↔ प्रचारित , incident ↔ घटना , different ↔ अलग
#2 Hindi | Error: 0.23
Mappings: For ↔ उदाहरण , example, ↔ उदाहरण , the ↔ की , 2009 ↔ 2009 , Hudson ↔ हडसन , River ↔ नदी , incident ↔ घटना , was ↔ था, , used ↔ इस्तेमाल , to ↔ को , solve ↔ हल , a ↔ एक , problem, ↔ समस्या , whereas ↔ जबकि , in ↔ में , other ↔ अन्य , cases, ↔ मामलों , like ↔ जैसे , UK ↔ यूके , riots, ↔ दंगों , social ↔ सोशल , media ↔ मीडिया , like ↔ जैसे , Twitter ↔ ट्विटर , and ↔ और , Facebook ↔ फेसबुक , made ↔ किया , situations ↔ स्थिति , worse ↔ खराब , by ↔ ने , organizing ↔ संगठित , mobs ↔ भीड़ , and ↔ और , spreading ↔ फैलाकर , messages ↔ संदेश , such ↔ जैसे , as ↔ जैसे , “let’s ↔ "चलो , go ↔ चलते , t